# Computing Perplexity with GPT-2 and Qwen 1B

In this notebook, we walk through how to compute the **perplexity** of a language model on a given text.

Perplexity measures how "surprised" a model is by the text. Lower perplexity means the model assigns higher probability to the text.

$$\text{PPL}(W) = \exp\left(-\frac{1}{N}\sum_{i=1}^{N} \log P(w_i \mid w_{<i})\right)$$

where $N$ is the number of tokens, and $P(w_i \mid w_{<i})$ is the model's predicted probability for token $w_i$ given all preceding tokens.

## Step 1: Setup and Imports

In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np

## Step 2: Define the Input Text

We will use a paragraph about the early history of the University of Chicago.

In [2]:
text = (
    "William Rainey Harper became the university's president on July 1, 1891, "
    "and classes first began on October 1, 1892. Harper offered large salaries "
    "to attract senior faculty, and in two years had a faculty of 120, including "
    "eight former university or college presidents. The undergraduate program was "
    "divided into two parts, with the first two years making up the Academic College, "
    "focusing on preparation for higher learning, and the last two years comprising "
    "the University College, with more advanced courses."
)
print(text)

William Rainey Harper became the university's president on July 1, 1891, and classes first began on October 1, 1892. Harper offered large salaries to attract senior faculty, and in two years had a faculty of 120, including eight former university or college presidents. The undergraduate program was divided into two parts, with the first two years making up the Academic College, focusing on preparation for higher learning, and the last two years comprising the University College, with more advanced courses.


## Step 3: Load GPT-2

GPT-2 (124M parameters) was released by OpenAI in 2019. Let's load the model and tokenizer.

In [3]:
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
gpt2_model = AutoModelForCausalLM.from_pretrained("gpt2")
gpt2_model.eval()  # set to evaluation mode (disables dropout)
print(f"GPT-2 parameters: {sum(p.numel() for p in gpt2_model.parameters()):,}")

GPT-2 parameters: 124,439,808


## Step 4: Tokenize the Text

Before we can feed text to a model, we need to convert it into token IDs. Let's see what tokens GPT-2 produces.

**Note:** GPT-2 does **not** have a start/BOS token. Its only special token is `<|endoftext|>` (EOS). So the first token ("William") has no preceding context. This is fine because when we compute perplexity, we only score tokens 1..N-1 (each conditioned on preceding tokens), skipping the first token.

In [4]:
gpt2_inputs = gpt2_tokenizer(text, return_tensors="pt")
gpt2_input_ids = gpt2_inputs["input_ids"]

print(f"Number of tokens: {gpt2_input_ids.shape[1]}")
print()

# Show each token and its ID
tokens = gpt2_input_ids[0].tolist()
print("Token ID -> Token string:")
for i, tok_id in enumerate(tokens):
    tok_str = gpt2_tokenizer.decode([tok_id])
    print(f"  [{i:2d}] {tok_id:6d} -> {repr(tok_str)}")

Number of tokens: 96

Token ID -> Token string:
  [ 0]  17121 -> 'William'
  [ 1]  10301 -> ' Rain'
  [ 2]   2959 -> 'ey'
  [ 3]  12686 -> ' Harper'
  [ 4]   2627 -> ' became'
  [ 5]    262 -> ' the'
  [ 6]   6403 -> ' university'
  [ 7]    338 -> "'s"
  [ 8]   1893 -> ' president'
  [ 9]    319 -> ' on'
  [10]   2901 -> ' July'
  [11]    352 -> ' 1'
  [12]     11 -> ','
  [13]   1248 -> ' 18'
  [14]   6420 -> '91'
  [15]     11 -> ','
  [16]    290 -> ' and'
  [17]   6097 -> ' classes'
  [18]    717 -> ' first'
  [19]   2540 -> ' began'
  [20]    319 -> ' on'
  [21]   3267 -> ' October'
  [22]    352 -> ' 1'
  [23]     11 -> ','
  [24]   1248 -> ' 18'
  [25]   5892 -> '92'
  [26]     13 -> '.'
  [27]  12686 -> ' Harper'
  [28]   4438 -> ' offered'
  [29]   1588 -> ' large'
  [30]  17058 -> ' salaries'
  [31]    284 -> ' to'
  [32]   4729 -> ' attract'
  [33]   4664 -> ' senior'
  [34]  12829 -> ' faculty'
  [35]     11 -> ','
  [36]    290 -> ' and'
  [37]    287 -> ' in'
  [38]    73

In [5]:
gpt2_input_ids.shape

torch.Size([1, 96])

## Step 5: Get Model Logits (Raw Predictions)

A causal language model outputs **logits** for each position: a vector of scores over the entire vocabulary.

For position $i$, the logits predict the distribution over the **next** token $w_{i+1}$.

So `logits[0]` predicts what comes after the first token, `logits[1]` predicts what comes after the second token, etc.

In [6]:
with torch.no_grad():
    gpt2_outputs = gpt2_model(**gpt2_inputs)

gpt2_logits = gpt2_outputs.logits  # shape: (batch=1, seq_len, vocab_size)
print(f"Logits shape: {gpt2_logits.shape}")
print(f"Vocabulary size: {gpt2_logits.shape[-1]}")
print(f"Sequence length: {gpt2_logits.shape[1]}")

Logits shape: torch.Size([1, 96, 50257])
Vocabulary size: 50257
Sequence length: 96


In [8]:
gpt2_outputs

CausalLMOutputWithCrossAttentions(loss=None, logits=tensor([[[ -35.3131,  -34.7376,  -38.1235,  ...,  -43.3306,  -42.1084,
           -34.7868],
         [ -99.3624,  -98.0258, -102.5517,  ..., -109.9021, -106.4707,
           -98.2294],
         [ -77.8806,  -76.7165,  -81.2766,  ...,  -87.8863,  -82.6802,
           -76.0936],
         ...,
         [-140.0932, -136.5210, -144.1208,  ..., -142.0854, -146.6803,
          -139.2219],
         [-125.2089, -126.0966, -134.3941,  ..., -135.5287, -133.1864,
          -125.4403],
         [-129.1685, -126.4687, -129.2262,  ..., -139.5148, -141.8782,
          -120.8957]]]), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=None, attentions=None, cross_attentions=None)

## Step 6: Convert Logits to Probabilities

We apply **softmax** to convert logits into a probability distribution:

$$P(w \mid \text{context}) = \frac{\exp(\text{logit}_w)}{\sum_{w'} \exp(\text{logit}_{w'})}$$

Then we extract the probability assigned to the **actual next token** at each position.

In [10]:
# logits[:, :-1, :] gives predictions for positions 0..N-2 (predicting tokens 1..N-1)
# input_ids[:, 1:] gives the actual tokens at positions 1..N-1 (the targets)
shift_logits = gpt2_logits[:, :-1, :]  # predictions
shift_labels = gpt2_input_ids[:, 1:]    # targets

print(f"Predictions shape: {shift_logits.shape}  (predicting {shift_logits.shape[1]} tokens)")
print(f"Targets shape:     {shift_labels.shape}")

Predictions shape: torch.Size([1, 95, 50257])  (predicting 95 tokens)
Targets shape:     torch.Size([1, 95])


In [11]:
# Convert logits to log-probabilities (more numerically stable than softmax + log)
# log_probs shape: (1, seq_len-1, vocab_size) - one distribution per position
log_probs = F.log_softmax(shift_logits, dim=-1)

# We want to extract the log-probability of the ACTUAL next token at each position.
# log_probs has shape (1, 95, 50257) - for each of 95 positions, a score for every vocab token
# shift_labels has shape (1, 95) - the actual token ID at each position
#
# torch.gather requires the index to have the same number of dimensions as the input.
# So we unsqueeze(-1) to go from (1, 95) -> (1, 95, 1), adding a trailing dimension.
# gather(dim=-1, ...) then picks one value per position along the vocab dimension.
# Result after gather: (1, 95, 1) - one log-prob per position, but with an extra dim.
# squeeze(-1) removes the trailing dimension: (1, 95, 1) -> (1, 95)
token_log_probs = log_probs.gather(dim=-1, index=shift_labels.unsqueeze(-1)).squeeze(-1)

print(f"Per-token log probabilities shape: {token_log_probs.shape}")
print(f"\nFirst 10 token log-probs: {token_log_probs[0, :10].tolist()}")

Per-token log probabilities shape: torch.Size([1, 95])

First 10 token log-probs: [-10.523319244384766, -0.8487353920936584, -10.820476531982422, -7.061079025268555, -1.024226188659668, -9.298816680908203, -0.2119549661874771, -2.533813953399658, -3.1634228229522705, -2.8085179328918457]


## Step 7: Inspect Per-Token Surprisal

**Surprisal** (negative log-probability) tells us how "surprised" the model is at each token. Higher surprisal means the model found that token less predictable.

$$\text{surprisal}(w_i) = -\log P(w_i \mid w_{<i})$$

In [12]:
# Show per-token surprisal
# Negate the tensor BEFORE converting to list (can't negate a Python list)
surprisals = (-token_log_probs[0]).tolist()

print(f"{'Position':>8}  {'Token':>15}  {'Log-Prob':>10}  {'Surprisal':>10}  {'Prob':>10}")
print("-" * 65)
for i, (tok_id, surp) in enumerate(zip(tokens[1:], surprisals)):
    tok_str = gpt2_tokenizer.decode([tok_id])
    log_p = -surp
    prob = np.exp(log_p)
    print(f"{i+1:>8}  {repr(tok_str):>15}  {log_p:>10.4f}  {surp:>10.4f}  {prob:>10.6f}")

Position            Token    Log-Prob   Surprisal        Prob
-----------------------------------------------------------------
       1          ' Rain'    -10.5233     10.5233    0.000027
       2             'ey'     -0.8487      0.8487    0.427956
       3        ' Harper'    -10.8205     10.8205    0.000020
       4        ' became'     -7.0611      7.0611    0.000858
       5           ' the'     -1.0242      1.0242    0.359074
       6    ' university'     -9.2988      9.2988    0.000092
       7             "'s"     -0.2120      0.2120    0.809001
       8     ' president'     -2.5338      2.5338    0.079356
       9            ' on'     -3.1634      3.1634    0.042281
      10          ' July'     -2.8085      2.8085    0.060294
      11             ' 1'     -2.1485      2.1485    0.116653
      12              ','     -0.3928      0.3928    0.675166
      13            ' 18'     -4.1264      4.1264    0.016141
      14             '91'     -4.5166      4.5166    0.010926
    

## Step 8: Compute Perplexity and Bits Per Character for GPT-2

Perplexity is the exponential of the average negative log-likelihood:

$$\text{PPL} = \exp\left(-\frac{1}{N}\sum_{i=1}^{N} \log P(w_i \mid w_{<i})\right) = \exp(\text{average surprisal})$$

**Problem with perplexity:** Different tokenizers produce different numbers of tokens for the same text, so perplexity (per-token average) is not directly comparable across models with different tokenizers.

**Bits per character (BPC)** normalizes by the number of *characters* instead of tokens, making it tokenizer-independent:

$$\text{BPC} = \frac{-\sum_{i=1}^{N} \log_2 P(w_i \mid w_{<i})}{|\text{characters}|} = \frac{\text{total NLL}}{\ln 2 \cdot |\text{characters}|}$$

The conversion from nats to bits is: $\log_2(x) = \frac{\ln(x)}{\ln 2}$

In [13]:
# Average negative log-likelihood
avg_nll = -token_log_probs[0].mean().item()

# Perplexity = exp(average NLL)
gpt2_ppl = np.exp(avg_nll)

# Bits per character (BPC)
# Total NLL is in nats; divide by ln(2) to convert to bits, then divide by number of characters
num_chars = len(text)
total_nll = -token_log_probs[0].sum().item()
gpt2_bpc = total_nll / (np.log(2) * num_chars)

print(f"GPT-2 Results:")
print(f"  Number of tokens scored: {token_log_probs.shape[1]}")
print(f"  Number of characters:   {num_chars}")
print(f"  Total NLL (nats):       {total_nll:.4f}")
print(f"  Average NLL:            {avg_nll:.4f}")
print(f"  Perplexity:             {gpt2_ppl:.2f}")
print(f"  Bits per character:     {gpt2_bpc:.4f}")

GPT-2 Results:
  Number of tokens scored: 95
  Number of characters:   511
  Total NLL (nats):       311.2673
  Average NLL:            3.2765
  Perplexity:             26.48
  Bits per character:     0.8788


## Step 9: Load Qwen 1B

Now let's compare with a larger, more recent model: Qwen2.5-0.5B (released by Alibaba). It has roughly 0.5B parameters and was trained on more data than GPT-2.

We expect it to achieve lower perplexity (less surprised by the text).

In [14]:
qwen_model_name = "Qwen/Qwen2.5-0.5B"

qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(qwen_model_name)
qwen_model.eval()
print(f"Qwen parameters: {sum(p.numel() for p in qwen_model.parameters()):,}")

Qwen parameters: 494,032,768


## Step 10: Tokenize with Qwen's Tokenizer

Different models use different tokenizers. Let's compare how Qwen tokenizes the same text.

In [15]:
qwen_inputs = qwen_tokenizer(text, return_tensors="pt")
qwen_input_ids = qwen_inputs["input_ids"]

print(f"GPT-2 tokens: {gpt2_input_ids.shape[1]}")
print(f"Qwen tokens:  {qwen_input_ids.shape[1]}")
print()

# Show Qwen's tokenization
qwen_tokens = qwen_input_ids[0].tolist()
print("Qwen Token ID -> Token string:")
for i, tok_id in enumerate(qwen_tokens):
    tok_str = qwen_tokenizer.decode([tok_id])
    print(f"  [{i:2d}] {tok_id:6d} -> {repr(tok_str)}")

GPT-2 tokens: 96
Qwen tokens:  107

Qwen Token ID -> Token string:
  [ 0]  44787 -> 'William'
  [ 1]  21911 -> ' Rain'
  [ 2]   1195 -> 'ey'
  [ 3]  32007 -> ' Harper'
  [ 4]   6116 -> ' became'
  [ 5]    279 -> ' the'
  [ 6]  12103 -> ' university'
  [ 7]    594 -> "'s"
  [ 8]   4767 -> ' president'
  [ 9]    389 -> ' on'
  [10]   5768 -> ' July'
  [11]    220 -> ' '
  [12]     16 -> '1'
  [13]     11 -> ','
  [14]    220 -> ' '
  [15]     16 -> '1'
  [16]     23 -> '8'
  [17]     24 -> '9'
  [18]     16 -> '1'
  [19]     11 -> ','
  [20]    323 -> ' and'
  [21]   6846 -> ' classes'
  [22]   1156 -> ' first'
  [23]   6009 -> ' began'
  [24]    389 -> ' on'
  [25]   6527 -> ' October'
  [26]    220 -> ' '
  [27]     16 -> '1'
  [28]     11 -> ','
  [29]    220 -> ' '
  [30]     16 -> '1'
  [31]     23 -> '8'
  [32]     24 -> '9'
  [33]     17 -> '2'
  [34]     13 -> '.'
  [35]  32007 -> ' Harper'
  [36]   8900 -> ' offered'
  [37]   3460 -> ' large'
  [38]  36432 -> ' salaries'
  [39] 

## Step 11: Compute Qwen's Perplexity

We repeat the same process: get logits, extract log-probabilities for the actual tokens, and compute perplexity.

In [16]:
with torch.no_grad():
    qwen_outputs = qwen_model(**qwen_inputs)

qwen_logits = qwen_outputs.logits
print(f"Qwen logits shape: {qwen_logits.shape}")
print(f"Qwen vocabulary size: {qwen_logits.shape[-1]}")

Qwen logits shape: torch.Size([1, 107, 151936])
Qwen vocabulary size: 151936


In [17]:
# Same shifting: predictions for tokens 1..N-1
qwen_shift_logits = qwen_logits[:, :-1, :]
qwen_shift_labels = qwen_input_ids[:, 1:]

# Log-probabilities of actual next tokens
# Same unsqueeze/squeeze pattern as GPT-2:
#   unsqueeze(-1): (1, N-1) -> (1, N-1, 1) so gather can index into vocab dimension
#   squeeze(-1):   (1, N-1, 1) -> (1, N-1) to remove the extra dimension after gathering
qwen_log_probs = F.log_softmax(qwen_shift_logits, dim=-1)
qwen_token_log_probs = qwen_log_probs.gather(
    dim=-1, index=qwen_shift_labels.unsqueeze(-1)
).squeeze(-1)

# Per-token surprisal for Qwen
qwen_surprisals = (-qwen_token_log_probs[0]).tolist()

print(f"{'Position':>8}  {'Token':>15}  {'Log-Prob':>10}  {'Surprisal':>10}  {'Prob':>10}")
print("-" * 65)
for i, (tok_id, surp) in enumerate(zip(qwen_tokens[1:], qwen_surprisals)):
    tok_str = qwen_tokenizer.decode([tok_id])
    log_p = -surp
    prob = np.exp(log_p)
    print(f"{i+1:>8}  {repr(tok_str):>15}  {log_p:>10.4f}  {surp:>10.4f}  {prob:>10.6f}")

Position            Token    Log-Prob   Surprisal        Prob
-----------------------------------------------------------------
       1          ' Rain'     -8.2670      8.2670    0.000257
       2             'ey'     -0.6674      0.6674    0.513054
       3        ' Harper'     -3.1014      3.1014    0.044987
       4        ' became'     -6.4550      6.4550    0.001573
       5           ' the'     -0.9279      0.9279    0.395381
       6    ' university'     -8.4925      8.4925    0.000205
       7             "'s"     -0.5060      0.5060    0.602901
       8     ' president'     -2.9363      2.9363    0.053059
       9            ' on'     -1.9406      1.9406    0.143614
      10          ' July'     -2.5643      2.5643    0.076977
      11              ' '     -0.0029      0.0029    0.997106
      12              '1'     -0.5571      0.5571    0.572894
      13              ','     -0.8291      0.8291    0.436448
      14              ' '     -0.0286      0.0286    0.971834
    

In [18]:
# Compute Qwen's perplexity and BPC
qwen_avg_nll = -qwen_token_log_probs[0].mean().item()
qwen_ppl = np.exp(qwen_avg_nll)

qwen_total_nll = -qwen_token_log_probs[0].sum().item()
qwen_bpc = qwen_total_nll / (np.log(2) * num_chars)  # num_chars is the same text

print(f"Qwen Results:")
print(f"  Number of tokens scored: {qwen_token_log_probs.shape[1]}")
print(f"  Number of characters:   {num_chars}")
print(f"  Total NLL (nats):       {qwen_total_nll:.4f}")
print(f"  Average NLL:            {qwen_avg_nll:.4f}")
print(f"  Perplexity:             {qwen_ppl:.2f}")
print(f"  Bits per character:     {qwen_bpc:.4f}")

Qwen Results:
  Number of tokens scored: 106
  Number of characters:   511
  Total NLL (nats):       259.9618
  Average NLL:            2.4525
  Perplexity:             11.62
  Bits per character:     0.7339


## Step 12: Compare the Two Models

In [19]:
print(f"{'Model':<15} {'Params':>10} {'Tokens':>8} {'Avg NLL':>10} {'Perplexity':>12} {'BPC':>8}")
print("-" * 70)
print(f"{'GPT-2':<15} {'124M':>10} {token_log_probs.shape[1]:>8} {avg_nll:>10.4f} {gpt2_ppl:>12.2f} {gpt2_bpc:>8.4f}")
print(f"{'Qwen2.5-0.5B':<15} {'0.5B':>10} {qwen_token_log_probs.shape[1]:>8} {qwen_avg_nll:>10.4f} {qwen_ppl:>12.2f} {qwen_bpc:>8.4f}")
print()
print(f"Characters in text: {num_chars}")
print()
print("Note: Perplexity is per-token, so it depends on the tokenizer.")
print("BPC is per-character, making it comparable across different tokenizers.")

Model               Params   Tokens    Avg NLL   Perplexity      BPC
----------------------------------------------------------------------
GPT-2                 124M       95     3.2765        26.48   0.8788
Qwen2.5-0.5B          0.5B      106     2.4525        11.62   0.7339

Characters in text: 511

Note: Perplexity is per-token, so it depends on the tokenizer.
BPC is per-character, making it comparable across different tokenizers.


## Step 13: Reusable Perplexity Function

Let's package everything into a clean function.

In [20]:
def compute_perplexity(model, tokenizer, text):
    """Compute perplexity and BPC of a causal LM on a given text."""
    inputs = tokenizer(text, return_tensors="pt")
    input_ids = inputs["input_ids"]

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits

    # Shift: logits[:-1] predict targets[1:]
    shift_logits = logits[:, :-1, :]
    shift_labels = input_ids[:, 1:]

    # Cross-entropy loss per token
    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_log_probs = log_probs.gather(dim=-1, index=shift_labels.unsqueeze(-1)).squeeze(-1)

    avg_nll = -token_log_probs.mean().item()
    total_nll = -token_log_probs.sum().item()
    ppl = np.exp(avg_nll)
    bpc = total_nll / (np.log(2) * len(text))

    return {
        "perplexity": ppl,
        "bpc": bpc,
        "avg_nll": avg_nll,
        "num_tokens": token_log_probs.shape[1],
        "token_log_probs": token_log_probs[0].tolist(),
    }

In [21]:
# Verify our function gives the same results
gpt2_result = compute_perplexity(gpt2_model, gpt2_tokenizer, text)
qwen_result = compute_perplexity(qwen_model, qwen_tokenizer, text)

print(f"GPT-2 perplexity:  {gpt2_result['perplexity']:.2f},  BPC: {gpt2_result['bpc']:.4f}")
print(f"Qwen perplexity:   {qwen_result['perplexity']:.2f},  BPC: {qwen_result['bpc']:.4f}")

GPT-2 perplexity:  26.48,  BPC: 0.8788
Qwen perplexity:   11.62,  BPC: 0.7339


## Step 14: Experiment - Try Your Own Text

Try computing perplexity on different types of text. What do you expect?
- Common English sentences (low perplexity)
- Rare or technical text (higher perplexity)
- Random characters (very high perplexity)

In [22]:
test_texts = [
    "The cat sat on the mat.",
    "The mitochondria is the powerhouse of the cell.",
    "Colorless green ideas sleep furiously.",
    "asdf jkl qwerty zxcv bnm poiu",
]

print(f"{'Text':<50} {'GPT-2 PPL':>10} {'GPT-2 BPC':>10} {'Qwen PPL':>10} {'Qwen BPC':>10}")
print("-" * 95)
for t in test_texts:
    g = compute_perplexity(gpt2_model, gpt2_tokenizer, t)
    q = compute_perplexity(qwen_model, qwen_tokenizer, t)
    display_text = t[:47] + "..." if len(t) > 50 else t
    print(f"{display_text:<50} {g['perplexity']:>10.2f} {g['bpc']:>10.4f} {q['perplexity']:>10.2f} {q['bpc']:>10.4f}")

Text                                                GPT-2 PPL  GPT-2 BPC   Qwen PPL   Qwen BPC
-----------------------------------------------------------------------------------------------
The cat sat on the mat.                                 90.24     1.6945      19.24     1.1128
The mitochondria is the powerhouse of the cell.         47.08     1.0641       8.77     0.5997
Colorless green ideas sleep furiously.                6413.28     1.9969      61.53     1.0948
asdf jkl qwerty zxcv bnm poiu                          789.08     4.3142     217.42     2.9451


## Key Takeaways

1. **Perplexity** measures how well a language model predicts a given text. Lower is better.
2. The computation involves: tokenize -> get logits -> softmax -> extract target token probabilities -> average NLL -> exponentiate.
3. The **shift by one** alignment is critical: logits at position $i$ predict the token at position $i+1$.
4. Different tokenizers produce different token counts, so **perplexity** (per-token average) depends on the tokenizer and is not directly comparable across models.
5. **Bits per character (BPC)** normalizes by character count instead of token count, making it tokenizer-independent and comparable across models.
6. Larger models trained on more data generally achieve lower perplexity and lower BPC.